In [0]:
%run /Workspace/Data_Ingestion_Atlikon/utilities

In [0]:
print(bronze_schema)
print(silver_schema)
print(gold_schema)

bronze
silver
gold


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://childcompanydata-myfirstproject/{data_source}/*.csv"
print(base_path)


s3://childcompanydata-myfirstproject/products/*.csv


In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load(base_path)\
    .withColumn("read_timestamp", F.current_timestamp())\
    .select("*", "_metadata.file_name", "_metadata.file_size")
    
df.write.format("delta")\
    .mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
display(df.limit(10))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2025-12-15T17:07:53.583Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2025-12-15T17:07:53.583Z,products.csv,1388


## Prepare Silver layer

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze.limit(10))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,energy bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,energy bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,energy bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (45g),25891201,protien bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (55g),25891202,protien bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Protien Bar Peanut Crunch (65g),25891203,protien bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,granola & cereals,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,granola & cereals,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,granola & cereals,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,recovery dairy,2025-12-15T17:07:43.833Z,products.csv,1388


#Transformations
- Drop Duplicates
- Title Case fix
- Fix Spelling mistake Protien to Protein

In [0]:
print("Rows before dropping duplicates: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates()
print("Rows before dropping duplicates: ", df_silver.count())

Rows before dropping duplicates:  20
Rows before dropping duplicates:  18


In [0]:
df_silver.select("category").distinct().show()

+-----------------+
|         category|
+-----------------+
|      energy bars|
|     protien bars|
|granola & cereals|
|   recovery dairy|
|   healthy snacks|
|  electrolyte mix|
+-----------------+



In [0]:
df_silver = df_silver\
            .withColumn("category", 
                        F.when(F.col("category").isNull(), None)
                         .otherwise(F.initcap("category")))

In [0]:
df_silver.select("category").distinct().show()

+-----------------+
|         category|
+-----------------+
|      Energy Bars|
|     Protien Bars|
|Granola & Cereals|
|   Recovery Dairy|
|   Healthy Snacks|
|  Electrolyte Mix|
+-----------------+



In [0]:
df_silver =df_silver\
    .withColumn("category", F.regexp_replace(F.col("category"), "(?i)Protien", "Protein"))\
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein"))

In [0]:
display(df_silver.limit(10))

product_name,product_id,category,read_timestamp,file_name,file_size
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protein Bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protein Bars,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2025-12-15T17:07:43.833Z,products.csv,1388
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2025-12-15T17:07:43.833Z,products.csv,1388


# Standardize Customer Attributes to Match Parent Company Data Model
- Add division column
- Varient column
- Create Product code column

In [0]:
category_mapping = {"Energy Bars":       "Nutrition Bars",
 "Protein Bars":       "Nutrition Bars",
 "Granola & Cereals":  "Breakfast Foods",
 "Recovery Dairy":     "Dairy & Recovery",
 "Healthy Snacks":     "Healthy Snacks",
 "Electrolyte Mix":    "Hydration & Electrolytes"}

df_category_mapping = spark.createDataFrame(
     [(k,v) for k,v in category_mapping.items()],
     ["category", "new_category"]
 )

display(df_category_mapping)

df_silver =df_silver.join(df_category_mapping, on="category", how="left").withColumnRenamed("new_category", "division")
display(df_silver.limit(10))

category,new_category
Energy Bars,Nutrition Bars
Protein Bars,Nutrition Bars
Granola & Cereals,Breakfast Foods
Recovery Dairy,Dairy & Recovery
Healthy Snacks,Healthy Snacks
Electrolyte Mix,Hydration & Electrolytes


category,product_name,product_id,read_timestamp,file_name,file_size,division
Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (120g),25891402,2025-12-15T17:07:43.833Z,products.csv,1388,Dairy & Recovery
Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (5 Sachets),25891603,2025-12-15T17:07:43.833Z,products.csv,1388,Hydration & Electrolytes
Energy Bars,SportsBar Energy Bar Choco Fudge (40g),25891102,2025-12-15T17:07:43.833Z,products.csv,1388,Nutrition Bars
Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),25891301,2025-12-15T17:07:43.833Z,products.csv,1388,Breakfast Foods
Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),25891403,2025-12-15T17:07:43.833Z,products.csv,1388,Dairy & Recovery
Protein Bars,SportsBar Protein Bar Peanut Crunch (45g),25891201,2025-12-15T17:07:43.833Z,products.csv,1388,Nutrition Bars
Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),25891601,2025-12-15T17:07:43.833Z,products.csv,1388,Hydration & Electrolytes
Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (350g),XYZ123,2025-12-15T17:07:43.833Z,products.csv,1388,Healthy Snacks
Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (200g),25891401,2025-12-15T17:07:43.833Z,products.csv,1388,Dairy & Recovery
Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (500g),25891501,2025-12-15T17:07:43.833Z,products.csv,1388,Healthy Snacks


In [0]:
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)

In [0]:
df_silver = (df_silver\
    .withColumn("product_code",
                F.sha2(F.col("product_name").cast("string"), 256))
    .withColumn("product_id",
                F.when(F.col("product_id").cast("string").rlike("^[0-9]+$"), F.col("product_id").cast("string"))
                 .otherwise(F.lit("999999").cast("string")))
    )

In [0]:
display(df_silver.limit(5))
df_silver = df_silver.withColumnRenamed("product_name", "product")\
                    .select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")
display(df_silver.limit(5))

category,product_name,product_id,read_timestamp,file_name,file_size,division,variant,product_code
Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (120g),25891402,2025-12-15T17:07:43.833Z,products.csv,1388,Dairy & Recovery,120g,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49
Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (5 Sachets),25891603,2025-12-15T17:07:43.833Z,products.csv,1388,Hydration & Electrolytes,5 Sachets,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa
Energy Bars,SportsBar Energy Bar Choco Fudge (40g),25891102,2025-12-15T17:07:43.833Z,products.csv,1388,Nutrition Bars,40g,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),25891301,2025-12-15T17:07:43.833Z,products.csv,1388,Breakfast Foods,400g,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),25891403,2025-12-15T17:07:43.833Z,products.csv,1388,Dairy & Recovery,80g,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9


product_code,division,category,product,variant,product_id,read_timestamp,file_name,file_size
fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (120g),120g,25891402,2025-12-15T17:07:43.833Z,products.csv,1388
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,Hydration & Electrolytes,Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (5 Sachets),5 Sachets,25891603,2025-12-15T17:07:43.833Z,products.csv,1388
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g,25891102,2025-12-15T17:07:43.833Z,products.csv,1388
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g,25891301,2025-12-15T17:07:43.833Z,products.csv,1388
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g,25891403,2025-12-15T17:07:43.833Z,products.csv,1388


In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

# Gold Layer Prep

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")
display(df_gold.limit(5))

product_code,product_id,division,category,product,variant
fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,25891402,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (120g),120g
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,25891603,Hydration & Electrolytes,Electrolyte Mix,SportsBar Electrolyte Mix Lemon-Lime (5 Sachets),5 Sachets
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,25891102,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,25891301,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,25891403,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g


In [0]:
df_gold.write\
 .format("delta") \
 .option("mergeSchema", "true") \
 .option("delta.enableChangeDataFeed", "true")\
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")